In [ ]:
# Lib & Dependencies
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , mean_absolute_error as mae
from sklearn.metrics import classification_report 
from sklearn.model_selection import *
import requests
from scipy import stats

import os

# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# import keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv1D, Flatten, Dropout, Input, BatchNormalization
from tensorflow.keras.layers import *
from tensorflow.keras import callbacks, optimizers
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from joblib import Parallel, delayed
from functools import partial
import gc
from tensorflow.keras.layers import MaxPooling1D
from scipy.stats import pearsonr
import warnings
import ast

warnings.simplefilter('ignore')

def scorer(y_test, y_pred):
    score = (stats.spearmanr(y_test, y_pred)*100)[0]
    return score

rmse = lambda y_t, y_p: mean_squared_error(y_t, y_p) ** 0.5

In [ ]:
DATA_PATH = '.'
train = pd.read_csv(DATA_PATH + os.sep + 'Train.csv')
test = pd.read_csv(DATA_PATH + os.sep + 'Test.csv')
ss = pd.read_csv(DATA_PATH + os.sep + 'SampleSubmission.csv')

train['is_train'] = 1
test['is_train'] = 0

ID_COL, TARGET_COL = "ID", 'Target'

In [ ]:
meta = pd.read_csv('metadata.csv')


In [ ]:
del train['is_train']
train.columns

Index(['ID', 'TransactionDates', 'PaymentsHistory', 'm1', 'm2', 'm3', 'm4',
       'm5', 'm6'],
      dtype='object')

In [ ]:

ID_COL, TARGET_COLS = 'ID', ['m1', 'm2', 'm3', 'm4', 'm5', 'm6']

train['is_train'], test['is_train'] = 1, 0

In [ ]:
df = train.append(test).reset_index(drop=True)
ID_COL, TARGET_COL = 'ID', ['m1', 'm2', 'm3', 'm4', 'm5', 'm6']

In [ ]:
df['month'] = df['TransactionDates'].apply(lambda x: [int(e.split('-')[0]) for e in ast.literal_eval(x)])
df['year'] = df['TransactionDates'].apply(lambda x: [int(e.split('-')[1]) for e in ast.literal_eval(x)])
df['PaymentsHistory'] = df['PaymentsHistory'].apply(lambda x: [float(e) for e in ast.literal_eval(x)])


In [ ]:
df['PaymentsHistory_cat'] = df['PaymentsHistory'].apply(lambda x: [e//5000 for e in x])

In [ ]:
tmp = np.array(df['PaymentsHistory_cat'].values.tolist())
np.unique(tmp).shape

(137,)

In [ ]:
def get_month_year(x):
  return [y*12 + m for y, m in zip(x['year'], x['month'])]

In [ ]:
df['month_year'] = df[['month', 'year']].apply(lambda x: get_month_year(x), axis=1)

In [ ]:
def get_cons_len_seq(x):
  return  [-1] * (41 - len(x)) + x

In [ ]:
df['month'] = df['month'].apply(lambda x: get_cons_len_seq(x))
df['year'] = df['year'].apply(lambda x: get_cons_len_seq(x))
df['PaymentsHistory'] = df['PaymentsHistory'].apply(lambda x: get_cons_len_seq(x))
df['PaymentsHistory_cat'] = df['PaymentsHistory_cat'].apply(lambda x: get_cons_len_seq(x))
df['month_year'] = df['month_year'].apply(lambda x: get_cons_len_seq(x))

In [ ]:
df['PaymentHistory_sum'] = df['PaymentsHistory'].apply(lambda x: [sum(x)] * 41)
df['month_year_range'] = df['PaymentsHistory'].apply(lambda x: [max(x) - min(x)] * 41)

In [ ]:
df['order'] = df['month_year'].apply(lambda x: np.argsort(x))

In [ ]:
for c in ['month', 'year', 'PaymentsHistory', 'month_year', 'PaymentsHistory_cat']:

  df[c] = df[[c, 'order']].apply(lambda x: list(np.array(x[c])[x['order']]), axis=1)

In [ ]:
def get_total_months_spent(x):
  res = []
  min_val = np.min([e for e in x if e != -1])


  for e in x:
    if e == -1:
      res.append(-1)
    else:
      res.append(e - min_val)

  return res

In [ ]:
df['total_months_spent_paying'] = df['month_year'].apply(lambda x: get_total_months_spent(x))

In [ ]:
def get_rank(x):
  i = 0
  res = []
  for e in x:
    if e == -1:
      res.append(e)
    else:
      i += 1
      res.append(i)
  return res

def get_paid_till_now(x):
  i = 0
  res = []
  curr_sum = 0
  for e in x:
    if e == -1:
      res.append(e)
    else:
      i += 1
      curr_sum += e
      res.append(curr_sum)
  return res

def get_remaining_till_now(x, total):
  i = 0
  res = []
  curr_sum = 0
  for e in x:
    if e == -1:
      res.append(total)
    else:
      i += 1
      curr_sum += e
      res.append(total - curr_sum)
  return res

In [ ]:
df['seq_order'] = df['month_year'].apply(lambda x: get_rank(x))
df['paid_till_now'] = df['PaymentsHistory'].apply(lambda x: get_paid_till_now(x))

In [ ]:
df['TotalContractValue'] = df['ID'].map(meta.set_index('ID')['TotalContractValue'])

In [ ]:
df['remaining_till_now'] = df[['PaymentsHistory', 'TotalContractValue']].\
apply(lambda x: get_remaining_till_now(x['PaymentsHistory'], x['TotalContractValue']), axis=1)

In [ ]:
df['month_year_diff'] = df['month_year'].apply(lambda x: [-1] + [x[i+1] - x[i] for i in range(len(x)-1)])

In [ ]:
cat_cols = ['month', 'year']

In [ ]:
df_nn = df[[ID_COL] + TARGET_COLS + cat_cols + ['is_train']]

df_nn['num_1'] = df['PaymentsHistory']
df_nn['num_2'] = df['month']
df_nn['num_3'] = df['year']
df_nn['num_4'] = df['month_year']
df_nn['num_5'] = df['remaining_till_now']
df_nn['num_6'] = df['paid_till_now']
df_nn['num_7'] = df['total_months_spent_paying']
# df_nn['num_5'] = df['seq_order']
# df_nn['num_5'] = df['PaymentHistory_sum']
# df_nn['num_6'] = df['month_year_range']

num_count = 6


# df_nn['num_5'] = df['month_year_diff']
df_nn['cat_1'] = df['month']
df_nn['cat_2'] = df['year']
df_nn['cat_3'] = df['PaymentsHistory_cat']

# for c in ['month', 'year', 'month_year']:
#   tmp = pd.Series(np.array(df[c].values.tolist()).flatten()).value_counts().to_dict()
#   df[c + '_freq'] = df[c].apply(lambda x: [tmp[e] for e in x])
#   df_nn[f'num_{num_count}'] = df[c + '_freq']
#   num_count += 1


# df_nn['cat_3'] = df['month_year']

In [ ]:
arr = np.array(df_nn['num_1'].values.tolist())

min_arr, max_arr = np.min(arr), np.max(arr)
range_arr = max_arr - min_arr

df_nn['num_1'] = df_nn['num_1'].apply(lambda x: [(e - min_arr)/(max_arr - min_arr) for e in x])

In [ ]:
use_cols = [c for c in df_nn.columns if 'num_' in c or 'cat_' in c and 'Feature' not in c and 'feature' not in c]
cat_cols = [c for c in use_cols if 'cat' in c]
num_cols = [c for c in use_cols if 'num' in c]
use_cols = num_cols + cat_cols

N_FEATS = len(use_cols)

cat_cols_unq_dct = {}

for c in cat_cols:
  unq_vals = np.unique(np.array(df_nn[c].values.tolist()))
  dct = pd.Series(index = unq_vals, data = np.arange(len(unq_vals))).to_dict()

  df_nn[c] = df_nn[c].apply(lambda x: [dct[e] for e in x])


for c in cat_cols:
  cat_cols_unq_dct[c] = np.unique(np.array(df_nn[c].values.tolist())).shape[0]

cat_cols_unq_dct

{'cat_1': 13, 'cat_2': 8, 'cat_3': 10}

In [ ]:
# for c in cat_cols:
#   df_nn[c] = df_nn[c].apply(lambda x: [cat_cols_unq_dct[c] + 1 if e == -1 else e for e in x])

In [ ]:
train, test = df_nn[df_nn['is_train'] == 1].reset_index(drop=True), df_nn[df_nn['is_train'] == 0].reset_index(drop=True)
train = train.sample(frac=1, random_state=2).reset_index(drop=True)

In [ ]:
train_arr = np.array(train[use_cols].values.tolist()).transpose((0, 2, 1))
test_arr = np.array(test[use_cols].values.tolist()).transpose((0, 2, 1))

print(train_arr.shape, test_arr.shape)
WINDOW_SIZE = train_arr[0].shape[0]

(2250, 41, 10) (750, 41, 10)


In [ ]:
print(train_arr.shape, test_arr.shape)

(2250, 41, 10) (750, 41, 10)


In [ ]:
def scorer(y_true, y_pred):
  # score = []
  # for i in range(3):
  return rmse(y_true.flatten(), y_pred.flatten())
  # return np.mean(score)

In [ ]:
best_model = None

from tensorflow.keras.callbacks import Callback
from tensorflow.keras.models import load_model

class IntervalEvaluation(Callback):
    def __init__(self, validation_data=(), interval=10, fold = 1):
        super(Callback, self).__init__()

        self.interval = interval
        self.X_val, self.y_val = validation_data
        self.scores = []
        self.best_score = np.inf
        self.best_model = []
        self.fold = fold

    def on_epoch_end(self, epoch, logs={}):

      y_pred = self.model.predict(self.X_val, verbose=0, batch_size=64*64, use_multiprocessing=False)
      # tf.keras.models.save_model(self.model, f'model_fold_{self.fold}_epoch_{epoch}')
      # try:
      # print(y_pred)
      score = scorer(self.y_val, y_pred)
      # print(score)
      # except:
      #   score = np.nan

      # print(score, self.best_score)
      if score < self.best_score:
        self.best_score = score
        tf.keras.models.save_model(self.model, f'best_model', save_format="h5")

      if epoch % self.interval == 0:
            print(f"Epoch: {epoch:03d} - score: {np.round(score, 2)}   best_score: {np.round(self.best_score, 2)}")

In [ ]:

from tensorflow.keras.layers import Layer
class WeightedSum(Layer):
    """A custom keras layer to learn a weighted sum of tensors"""

    def __init__(self, **kwargs):
        super(WeightedSum, self).__init__(**kwargs)

    def build(self, input_shape=1):
        self.a = self.add_weight(
            name='alpha',
            shape=(),
            initializer='ones',
            dtype='float32',
            trainable=True)
        
        self.b = self.add_weight(
            name='beta',
            shape=(),
            initializer='ones',
            dtype='float32',
            trainable=True)
        
        self.c = self.add_weight(
            name='gamma',
            shape=(),
            initializer='ones',
            dtype='float32',
            trainable=True)

        super(WeightedSum, self).build(input_shape)

    def call(self, model_outputs):
        return self.a * model_outputs[0] + self.b * model_outputs[1] + self.c * model_outputs[2]

    def compute_output_shape(self, input_shape):
        return input_shape[0]

In [ ]:
def mean_absolute_error_keras(y_true, y_pred):
        return K.mean(K.abs(y_pred - y_true))
    
def root_mean_squared_error(y_true, y_pred):
        return K.sqrt(K.mean(K.square(y_pred - y_true))) 

def mcmae(y_actual, y_pred, num_scored=len(TARGET_COLS)):

    return root_mean_squared_error(Flatten()(y_actual), Flatten()(y_pred))

def make_model():
    
    NUMERICAL_INPUTS_CNT = len(use_cols) - len(cat_cols)

    inputs = Input(shape=(WINDOW_SIZE, N_FEATS))
    num_inputs = inputs[:, :, :NUMERICAL_INPUTS_CNT]
    cat_inputs = inputs[:, :, NUMERICAL_INPUTS_CNT:]
    initializer = tf.keras.initializers.glorot_uniform(seed=66)
  
    for i, c in enumerate(cat_cols):
      cat_inp = cat_inputs[:, :, i:i+1]
      embed = Embedding(input_dim=cat_cols_unq_dct[c]+2, output_dim=64, embeddings_initializer=initializer)(cat_inp)
      reshaped = tf.reshape(embed, shape=(-1, embed.shape[1],  embed.shape[2] * embed.shape[3]))
      reshaped = SpatialDropout1D(0.2, seed=2)(reshaped)
      num_inputs = concatenate([reshaped, num_inputs], axis=2)

    
    x = BatchNormalization()(num_inputs)
    x = concatenate([x, x], axis=2)

    orig_inps = x
    N_FILTERS = x.shape[2]
    skip_connection = orig_inps
    N_LAYERS = 17
    SKIP_CONNECTION_EVERY_N_LAYERS = 3

    for i in range(N_LAYERS):
      if i > 0 and i % SKIP_CONNECTION_EVERY_N_LAYERS  == 0:
        x = Conv1D(filters=N_FILTERS, kernel_size=(6,), strides=1, kernel_initializer=initializer, activation='relu',
               padding='same')(x)
        x = tf.keras.layers.LSTM(
    N_FILTERS,
    activation="tanh",
    recurrent_activation="sigmoid",
    use_bias=True,
    kernel_initializer="glorot_uniform",
    recurrent_initializer="orthogonal",
    bias_initializer="zeros",
    unit_forget_bias=True,
    kernel_regularizer=None,
    recurrent_regularizer=None,
    bias_regularizer=None,
    activity_regularizer=None,
    kernel_constraint=None,
    recurrent_constraint=None,
    bias_constraint=None,
    dropout=0.1,
    recurrent_dropout=0.0,
    return_sequences=True,
    return_state=False,
    go_backwards=False,
    stateful=False,
    time_major=False,
    unroll=False,
)(x)
        x = BatchNormalization()(x)
        x = tf.keras.layers.Add()([x, skip_connection, orig_inps])
        x = ReLU()(x)
        skip_connection = x
      else:
        x = tf.keras.layers.LSTM(
    N_FILTERS,
    activation="tanh",
    recurrent_activation="sigmoid",
    use_bias=True,
    kernel_initializer="glorot_uniform",
    recurrent_initializer="orthogonal",
    bias_initializer="zeros",
    unit_forget_bias=True,
    kernel_regularizer=None,
    recurrent_regularizer=None,
    bias_regularizer=None,
    activity_regularizer=None,
    kernel_constraint=None,
    recurrent_constraint=None,
    bias_constraint=None,
    dropout=0.1,
    recurrent_dropout=0.0,
    return_sequences=True,
    return_state=False,
    go_backwards=False,
    stateful=False,
    time_major=False,
    unroll=False,
)(x)
        x = BatchNormalization()(x)

      x = Dropout(0.07, seed=2)(x)

    x = MaxPool1D()(x)
    x = Flatten()(x)
    x = Dropout(0.07, seed=2)(x)
    out = Dense(6, kernel_initializer=initializer)(x)

    model = tf.keras.Model(inputs=inputs, outputs=out)
    
    optimizer=Adam(lr = 5e-4)
    model.compile(optimizer=optimizer, loss=mcmae)
    
    return model

m = make_model()
m.build((None, WINDOW_SIZE, N_FEATS))
m.count_params()
# m.summary()

26402112

In [ ]:
tf.random.set_seed(7)
N_SPLITS = 11
folds = KFold(N_SPLITS)
oofs = np.zeros((len(train), 6))
preds = np.zeros((len(test), 6))

fi_df = pd.DataFrame()

for fold_, (trn_idx, val_idx) in enumerate(folds.split(train, train['ID'])):

    X_trn, y_trn = train_arr[trn_idx], train[TARGET_COLS].iloc[trn_idx].values
    X_val, y_val = train_arr[val_idx], train[TARGET_COLS].iloc[val_idx].values

    es = tf.keras.callbacks.EarlyStopping(patience=30, 
                                          verbose=1, 
                                          restore_best_weights=True)
      
    print_val_results = IntervalEvaluation((X_val, y_val), interval=5, fold=fold_)

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.4, patience=6,verbose=0,
                                                      mode="auto", min_delta=0.0001, cooldown=0, min_lr=0)

    clf = make_model()
    history = clf.fit(X_trn,
          y_trn,
          epochs=150,
          verbose=0,
          batch_size=32,
          validation_data=(X_val, y_val),
          callbacks=[reduce_lr, es, print_val_results],
          )
    
    # break

    best_model = load_model('best_model', custom_objects={'mcmae': mcmae, 'WeightedSum': WeightedSum})
    vp = best_model.predict(X_val)
    val_score = round(scorer(y_val, vp), 5)

    corr_score = np.array([pearsonr(y_val[:, i], vp[:, i])[0] for i in range(3)]).mean()
    print(f'\nFold {fold_+1} Val score: {val_score}, Corr score: {corr_score} \n')
    

    tp = best_model.predict(test_arr)

    oofs[val_idx] = vp
    preds += tp/N_SPLITS


oof_score = round(scorer(train[TARGET_COLS].values, oofs), 5)
mae_score = oof_score.copy()

print(f'\nOOF Score is : {oof_score}')

Epoch: 000 - score: 1652.36   best_score: 1652.36
Epoch: 005 - score: 1276.84   best_score: 1276.84
Epoch: 010 - score: 1236.53   best_score: 1236.06
Epoch: 015 - score: 1231.43   best_score: 1219.85
Epoch: 020 - score: 1204.95   best_score: 1204.95
Epoch: 025 - score: 1212.7   best_score: 1195.93
Epoch: 030 - score: 1188.35   best_score: 1188.35
Epoch: 035 - score: 1191.2   best_score: 1187.76
Epoch: 040 - score: 1194.79   best_score: 1187.76
Epoch: 045 - score: 1185.61   best_score: 1185.61
Epoch: 050 - score: 1188.99   best_score: 1182.79
Epoch: 055 - score: 1186.03   best_score: 1182.79
Epoch: 060 - score: 1186.19   best_score: 1182.79
Epoch: 065 - score: 1184.87   best_score: 1182.79
Epoch: 070 - score: 1185.09   best_score: 1182.79
Epoch: 075 - score: 1185.12   best_score: 1182.79
Restoring model weights from the end of the best epoch.
Epoch 00078: early stopping

Fold 1 Val score: 1182.79317, Corr score: 0.5403960811885697 

Epoch: 000 - score: 1281.33   best_score: 1281.33
Epoc

In [ ]:
oofs_df = pd.DataFrame(oofs, columns = TARGET_COLS)
oofs_df[ID_COL] = train[ID_COL]
tmp = pd.melt(oofs_df, id_vars = [ID_COL], value_vars=TARGET_COLS, var_name='month', value_name='Target')
tmp['ID'] = tmp['ID'].astype('str') + ' x ' + tmp['month'].astype('str')
tmp.to_csv('paygo_lstm_cnn_v6_oofs.csv', index=False)

In [ ]:
preds_df = pd.DataFrame(preds, columns = TARGET_COLS)
preds_df[ID_COL] = test[ID_COL]
tmp = pd.melt(preds_df, id_vars = [ID_COL], value_vars=TARGET_COLS, var_name='month', value_name='Target')
tmp['ID'] = tmp['ID'].astype('str') + ' x ' + tmp['month'].astype('str')
ss['Target'] = ss[ID_COL].map(tmp.set_index(ID_COL)['Target'])
ss.to_csv('paygo_lstm_cnn_v6.csv', index=False)